# Explorar las bases de `data/` — práctica de pandas

Cuaderno de **práctica** (no es parte de la entrega). Sirve para entender cómo se
consultan los `.parquet` cacheados: cargar, inspeccionar, filtrar, contar y agrupar.

> Requiere haber corrido `00_setup.ipynb` antes (para que existan los parquet).

## 1. Cargar una base

Un `.parquet` es un DataFrame guardado en disco. Lo levantamos con `pd.read_parquet`.
`from helpers import *` ya nos trae `pd` y `DATA_DIR` listos.

In [ ]:
from helpers import *   # trae pd, DATA_DIR, MATCH_ARG_SAU, etc.

ev = pd.read_parquet(DATA_DIR / 'eventos_arg_sau.parquet')
# (equivale a:  ev = cargar_eventos(MATCH_ARG_SAU, 'arg_sau') )
type(ev)   # -> pandas.DataFrame

## 2. ¿Qué tengo? — inspección rápida

Cuatro comandos reflejo **antes** de filtrar nada:

In [ ]:
print('shape:', ev.shape)      # (3329, 93) -> (filas, columnas)
print('filas   :', ev.shape[0]) # 3329 eventos (cada fila = una acción)
print('columnas:', ev.shape[1]) # 93 variables por evento

ev.columns.tolist()[:25]        # primeros 25 nombres de columna

In [ ]:
# info() es EL atajo para 'qué información tiene esta tabla':
# lista columnas + cuántos valores no-nulos + tipo de dato de cada una.
ev.info()

In [ ]:
ev.head()   # ojear las primeras 5 filas

## 3. Una columna = Series · varias = DataFrame

Detalle clave de pandas (esto te confundía):
- **1 corchete** → `Series` (una sola columna, como una lista con índice).
- **2 corchetes** → `DataFrame` (sub-tabla con esas columnas).

In [ ]:
una = ev['player']                 # Series
varias = ev[['player', 'type']]    # DataFrame
print(type(una))
print(type(varias))
varias.head()

In [ ]:
# Ver UNA fila 'en vertical' (cómodo cuando hay 93 columnas):
# nombre_de_columna -> valor
ev.iloc[6]

## 4. Filtrar — quedarme con un subconjunto

Le pasamos a la tabla una condición True/False por fila.

In [ ]:
# Solo los pases
pases = ev[ev['type'] == 'Pass']
print('pases en el partido:', len(pases))

# Solo las acciones de Messi (contains es más robusto que el nombre exacto)
messi = ev[ev['player'].str.contains('Messi', na=False)]
print('acciones de Messi:', len(messi))
messi[['minute','type']].head()

## 5. Contar — `value_counts()`

Cuenta cuántas veces aparece cada valor. Responde preguntas tipo *'¿quién más...?'*

In [ ]:
# ¿Quién dio más pases? (top 5)
ev[ev['type'] == 'Pass']['player'].value_counts().head()

In [ ]:
# ¿Quién hizo más recuperaciones? (Punto 2 del proyecto)
ev[ev['type'] == 'Ball Recovery']['player'].value_counts().head()

## 6. Agrupar — `groupby()`

Parte la tabla en grupos y calcula algo por grupo. El corazón del análisis.

In [ ]:
# xG total por equipo (clave para el Punto 2)
ev.groupby('team')['shot_statsbomb_xg'].sum()

In [ ]:
# Acciones por tercio de la cancha (defensivo / medio / ofensivo)
ev2 = añadir_xy(ev)   # agrega columnas x, y desde 'location'
ev2['tercio'] = pd.cut(ev2['x'], [0, 40, 80, 120],
                       labels=['Defensivo','Medio','Ofensivo'])
ev2['tercio'].value_counts()

## 7. La tabla de partidos (otra base, otra granularidad)

Acá **cada fila es un partido** (no un evento). Sirve para el Punto 6.

In [ ]:
partidos = pd.read_parquet(DATA_DIR / 'partidos_mundial_2022.parquet')
print('partidos:', partidos.shape[0])
partidos[['match_id','home_team','away_team','home_score','away_score','match_date']].head()

## 8. Tu turno — practicá

Ideas para resolver vos solo (todas se hacen con lo de arriba):
1. ¿Cuántos **tiros** (`type == 'Shot'`) tuvo cada equipo?
2. ¿Quién tuvo más **intercepciones** (`Interception`)?
3. xG total de **Arabia Saudita** filtrando por equipo.
4. Cargá `eventos_arg_fra.parquet` y repetí el top de pasadores.

In [ ]:
# Escribí acá tu práctica:
